<a href="https://colab.research.google.com/github/Jyothidatla25/JYOTHIRMAI_INFO-5731/blob/main/INFO5731_Assignment_2_final_(1)_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **INFO5731 Assignment 2**

In this assignment, we will delve into various aspects of natural language processing (NLP) and text analysis. The tasks are designed to deepen your understanding of key NLP concepts and techniques, as well as to provide hands-on experience with practical applications.

Through these tasks, you'll gain practical experience in NLP techniques such as N-gram analysis, TF-IDF, word embedding model creation, and sentiment analysis dataset creation.

**Expectations**:
* Use the provided `.ipynb` document to write your code and respond to the questions. Avoid generating a new file.
* Write complete answers and run all the cells before submission.
* Make sure the submission is "clean"; i.e., no unnecessary code cells.
* Once finished, allow shared rights from the top right corner (see Canvas for details).
* **Note:** Use the same dataset you created in **Assignment 1** for **Questions 1–3**.

**Total points:** 100

**Deadline:** See Canvas

Late submission will have a penalty of **10% reduction for each day** after the deadline.


## Question 1 (25 points)

**Understand N-gram**

Write a **Python** program to conduct N-gram analysis based on the dataset you created in **Assignment 1**. You need to write **code from scratch instead of using any pre-existing libraries** to do so:

(1) Count the frequency of all the N-grams (**N = 3** and **N = 2**).

(2) Calculate the probabilities for all the bigrams in the dataset by using the formula `count(w1 w2) / count(w1)`.

For example, `count(really like) / count(really) = 1 / 3 = 0.33`.

(3) Extract all the noun phrases and calculate the relative probabilities of each review in terms of other reviews (abstracts, or tweets) by using the formula `frequency(noun phrase) / max frequency(noun phrase)` on the whole dataset. You may use NLP libraries (e.g., **spaCy** or **NLTK**) for noun phrase extraction.

Print out the result in a table with all noun phrases as the column names and all **100** reviews (abstracts, or tweets) as the row names.


In [ ]:


!pip -q install spacy
!python -m spacy download en_core_web_sm -q

import pandas as pd
import re
import spacy
from collections import defaultdict
from google.colab import files

# Upload your Assignment 1 CSV
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

text_column = "clean_text"

df_q1 = df[[text_column]].copy()
df_q1 = df_q1[df_q1[text_column].notna()]
df_q1[text_column] = df_q1[text_column].astype(str).str.strip()
df_q1 = df_q1[df_q1[text_column] != ""]
df_q1 = df_q1.head(100).reset_index(drop=True)
df_q1.index = [f"Review_{i+1}" for i in range(len(df_q1))]

print("Number of reviews used:", len(df_q1))

def clean_and_tokenize(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split()

df_q1["tokens"] = df_q1[text_column].apply(clean_and_tokenize)
df_q1 = df_q1[df_q1["tokens"].apply(len) > 0].copy()

def generate_ngrams(tokens, n):
    ngrams_list = []
    for i in range(len(tokens) - n + 1):
        ngrams_list.append(tuple(tokens[i:i+n]))
    return ngrams_list

unigram_counts = defaultdict(int)
bigram_counts = defaultdict(int)
trigram_counts = defaultdict(int)

for tokens in df_q1["tokens"]:
    for word in tokens:
        unigram_counts[word] += 1
    for bg in generate_ngrams(tokens, 2):
        bigram_counts[bg] += 1
    for tg in generate_ngrams(tokens, 3):
        trigram_counts[tg] += 1

bigram_freq_df = pd.DataFrame(
    [(f"{bg[0]} {bg[1]}", count) for bg, count in bigram_counts.items()],
    columns=["Bigram", "Frequency"]
).sort_values(by="Frequency", ascending=False).reset_index(drop=True)

trigram_freq_df = pd.DataFrame(
    [(f"{tg[0]} {tg[1]} {tg[2]}", count) for tg, count in trigram_counts.items()],
    columns=["Trigram", "Frequency"]
).sort_values(by="Frequency", ascending=False).reset_index(drop=True)

print("\nTop Bigrams:")
print(bigram_freq_df)

print("\nTop Trigrams:")
print(trigram_freq_df)

bigram_probabilities = []

for bg, bg_count in bigram_counts.items():
    w1, w2 = bg
    prob = bg_count / unigram_counts[w1]
    bigram_probabilities.append((f"{w1} {w2}", bg_count, unigram_counts[w1], round(prob, 4)))

bigram_prob_df = pd.DataFrame(
    bigram_probabilities,
    columns=["Bigram", "Bigram_Count", "Count_of_First_Word", "Probability"]
).sort_values(by="Probability", ascending=False).reset_index(drop=True)

print("\nBigram Probabilities:")
print(bigram_prob_df)

nlp = spacy.load("en_core_web_sm")

def extract_noun_phrases(text):
    doc = nlp(str(text))
    noun_phrases = []
    for chunk in doc.noun_chunks:
        np_text = chunk.text.lower().strip()
        np_text = re.sub(r"[^a-zA-Z\s]", " ", np_text)
        np_text = re.sub(r"\s+", " ", np_text).strip()
        if np_text != "":
            noun_phrases.append(np_text)
    return noun_phrases

df_q1["noun_phrases"] = df_q1[text_column].apply(extract_noun_phrases)

global_np_counts = defaultdict(int)
for np_list in df_q1["noun_phrases"]:
    for np_item in np_list:
        global_np_counts[np_item] += 1

max_np_freq = max(global_np_counts.values()) if len(global_np_counts) > 0 else 1
all_noun_phrases = sorted(global_np_counts.keys())

np_table = pd.DataFrame(0.0, index=df_q1.index, columns=all_noun_phrases)

for row_name, np_list in zip(df_q1.index, df_q1["noun_phrases"]):
    review_np_counts = defaultdict(int)
    for np_item in np_list:
        review_np_counts[np_item] += 1
    for np_item, freq in review_np_counts.items():
        np_table.loc[row_name, np_item] = round(freq / max_np_freq, 4)

print("\nNoun Phrase Relative Probability Table:")
print(np_table)

bigram_freq_df.to_csv("bigram_frequencies_q1.csv", index=False)
trigram_freq_df.to_csv("trigram_frequencies_q1.csv", index=False)
bigram_prob_df.to_csv("bigram_probabilities_q1.csv", index=False)
np_table.to_csv("noun_phrase_relative_probability_table_q1.csv")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 91.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## Question 2 (25 points)

**Understand TF-IDF and Document Representation**

Starting from the documents (all the reviews, abstracts, or tweets) collected for **Assignment 1**, write a **Python** program:

(1) Build the **document-term weight (`tf * idf`) matrix**.

(2) Rank the documents with respect to a query (design a query by yourself, for example, "An outstanding movie with a haunting performance and best character development") by using cosine similarity.

**Note:** You need to write **code from scratch instead of using any pre-existing libraries** to do so.


In [ ]:


import pandas as pd
import re
import math
from collections import defaultdict

try:
    df
    print("Using existing dataframe.")
except NameError:
    from google.colab import files
    uploaded = files.upload()
    file_name = list(uploaded.keys())[0]
    df = pd.read_csv(file_name)

text_column = "clean_text"

df_q2 = df[[text_column]].copy()
df_q2 = df_q2[df_q2[text_column].notna()]
df_q2[text_column] = df_q2[text_column].astype(str).str.strip()
df_q2 = df_q2[df_q2[text_column] != ""]
df_q2 = df_q2.head(100).reset_index(drop=True)
df_q2.index = [f"Doc_{i+1}" for i in range(len(df_q2))]

def clean_and_tokenize(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split()

df_q2["tokens"] = df_q2[text_column].apply(clean_and_tokenize)
df_q2 = df_q2[df_q2["tokens"].apply(len) > 0].copy()

vocabulary = sorted(set(token for tokens in df_q2["tokens"] for token in tokens))
print("Vocabulary size:", len(vocabulary))

document_frequency = {}

for term in vocabulary:
    count_docs = 0
    for tokens in df_q2["tokens"]:
        if term in tokens:
            count_docs += 1
    document_frequency[term] = count_docs

N = len(df_q2)
idf = {}

for term in vocabulary:
    df_term = document_frequency[term]
    idf[term] = math.log10(N / df_term) if df_term > 0 else 0.0

tf_idf_matrix = pd.DataFrame(0.0, index=df_q2.index, columns=vocabulary)

for doc_name, tokens in zip(df_q2.index, df_q2["tokens"]):
    term_counts = defaultdict(int)
    for token in tokens:
        term_counts[token] += 1
    for term, tf in term_counts.items():
        tf_idf_matrix.loc[doc_name, term] = tf * idf[term]

print("\nTF-IDF Matrix:")
print(tf_idf_matrix)

query = "natural language processing text cleaning and important steps"
print("\nQuery:", query)

query_tokens = clean_and_tokenize(query)
query_term_counts = defaultdict(int)

for token in query_tokens:
    query_term_counts[token] += 1

query_vector = []
for term in vocabulary:
    tf_query = query_term_counts[term]
    query_vector.append(tf_query * idf.get(term, 0.0))

def cosine_similarity(vec1, vec2):
    dot_product = 0.0
    norm1 = 0.0
    norm2 = 0.0

    for a, b in zip(vec1, vec2):
        dot_product += a * b
        norm1 += a * a
        norm2 += b * b

    norm1 = math.sqrt(norm1)
    norm2 = math.sqrt(norm2)

    if norm1 == 0 or norm2 == 0:
        return 0.0

    return dot_product / (norm1 * norm2)

document_scores = []

for doc_name in tf_idf_matrix.index:
    doc_vector = tf_idf_matrix.loc[doc_name].tolist()
    score = cosine_similarity(doc_vector, query_vector)
    original_text = df_q2.loc[doc_name, text_column]
    document_scores.append([doc_name, round(score, 4), original_text])

ranking_df = pd.DataFrame(document_scores, columns=["Document", "Cosine_Similarity", "Text"])
ranking_df = ranking_df.sort_values(by="Cosine_Similarity", ascending=False).reset_index(drop=True)

print("\nRanked Documents:")
print(ranking_df)

tf_idf_matrix.to_csv("tf_idf_matrix_q2.csv")
ranking_df.to_csv("document_ranking_q2.csv", index=False)

## Question 3 (25 points)

**Create your own word embedding model**

Use the data you collected for **Assignment 1** to build a word embedding model. You may use existing libraries (e.g., **gensim** or **transformers**) for training embeddings.

(1) Train a **300-dimensional** word embedding model (e.g., **Word2Vec, GloVe, ULMFiT, or a fine-tuned BERT model**).

(2) Visualize the embeddings using **PCA** or **t-SNE** in 2D. Create a scatter plot of at least **20 words** and show how similar words cluster together.

(3) Calculate the **cosine similarity** between a few pairs of words to examine whether the model captures semantic similarity accurately.

**References:**

- https://machinelearningmastery.com/develop-word-embeddings-python-gensim/
- https://jaketae.github.io/study/word2vec/


In [ ]:


!pip -q install gensim scipy

import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from gensim.models import Word2Vec

try:
    df
    print("Using existing dataframe.")
except NameError:
    from google.colab import files
    uploaded = files.upload()
    file_name = list(uploaded.keys())[0]
    df = pd.read_csv(file_name)

text_column = "clean_text"

df_q3 = df[[text_column]].copy()
df_q3 = df_q3[df_q3[text_column].notna()]
df_q3[text_column] = df_q3[text_column].astype(str).str.strip()
df_q3 = df_q3[df_q3[text_column] != ""].reset_index(drop=True)

def clean_and_tokenize(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split()

df_q3["tokens"] = df_q3[text_column].apply(clean_and_tokenize)
df_q3 = df_q3[df_q3["tokens"].apply(len) > 0].copy()

sentences = df_q3["tokens"].tolist()

model = Word2Vec(
    sentences=sentences,
    vector_size=300,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=20
)

print("Model trained successfully.")
print("Vocabulary size:", len(model.wv.index_to_key))

words_to_plot = model.wv.index_to_key[:20]
if len(words_to_plot) < 20:
    words_to_plot = model.wv.index_to_key[:len(model.wv.index_to_key)]

word_vectors = np.array([model.wv[word] for word in words_to_plot])

pca = PCA(n_components=2)
reduced_vectors = pca.fit_transform(word_vectors)

plt.figure(figsize=(12, 8))
plt.scatter(reduced_vectors[:, 0], reduced_vectors[:, 1])

for i, word in enumerate(words_to_plot):
    plt.annotate(word, (reduced_vectors[i, 0], reduced_vectors[i, 1]))

plt.title("2D Visualization of Word Embeddings using PCA")
plt.xlabel("PCA Dimension 1")
plt.ylabel("PCA Dimension 2")
plt.grid(True)
plt.show()

def cosine_similarity(vec1, vec2):
    numerator = np.dot(vec1, vec2)
    denominator = np.linalg.norm(vec1) * np.linalg.norm(vec2)
    if denominator == 0:
        return 0.0
    return numerator / denominator

vocab_words = model.wv.index_to_key

word_pairs = []
if len(vocab_words) >= 6:
    word_pairs = [
        (vocab_words[0], vocab_words[1]),
        (vocab_words[2], vocab_words[3]),
        (vocab_words[4], vocab_words[5])
    ]

print("\nCosine Similarity Between Word Pairs:")
for w1, w2 in word_pairs:
    sim = cosine_similarity(model.wv[w1], model.wv[w2])
    print(f"{w1} - {w2}: {sim:.4f}")

print("\nMost similar words:")
for word in vocab_words[:3]:
    print(f"\nTop 5 words similar to '{word}':")
    for similar_word, score in model.wv.most_similar(word, topn=5):
        print(f"{similar_word}: {score:.4f}")

model.save("word2vec_300d.model")

## Question 4 (20 Points)

**Create your own training and evaluation dataset for an NLP task.**

**You do not need to write a program for this question.**

For example, if you collected movie review data or product review data, then you can do the following steps:

* Read each review (abstract or tweet) you collected in detail, and annotate each review with a sentiment (**positive, negative, or neutral**).

* Save the annotated dataset into a **CSV** file with three columns (`document_id`, `clean_text`, `sentiment`), upload the CSV file to GitHub, and submit the file link below.

This dataset will be used for **Assignment 4: Sentiment Analysis and Text Classification**.


1. Which NLP task would you like to perform on your selected dataset (**NER, summarization, sentiment analysis, or text classification**)?
2. Explain the labeling schema you used and mention the labels.

3. You may use AI assistance for labeling the data only.


In [ ]:
https://github.com/Jyothidatla25/JYOTHIRMAI_INFO-5731

# Mandatory Question (5 Points)

Provide your thoughts on the assignment by filling this survey link. What did you find challenging, and what aspects did you enjoy? Your opinion on the provided time to complete the assignment.

In [ ]:
The assignment made me feel familiar with the major concepts of NLP which include N-grams, TF-IDF, word embeddings, and dataset annotation. The hardest aspect was applying the algorithms in their purest form, particularly generation of N-grams, computation of TF-IDF values, and cosine similarity since it contained delicate logic and debugging. It was also somewhat difficult to deal with mistakes in Colab and make sure that the code can run without problems.

I liked working on the word embedding model and visualising the results with the help of PCA because it helped to make the concepts more realistic and appealing. It was also handy to observe the way in which various techniques of NLP are interrelated and how they can be applied to actual data.

The assignment had a proper amount of time, yet it had to be done consistently, particularly in debugging and labeling the dataset. Generally, the assignment proved quite beneficial in getting the practical experience in NLP techniques.